In [ ]:
# pip install langchain langchain_community boto3 awscli faiss-cpu langchain_aws
# STEP 1 - Generate an External Knowledge base

from langchain_community.document_loaders import TextLoader

loader = TextLoader("C:/content/Budget_Speech.txt")
pages = loader.load()

print("Metadata: \n", pages[0].metadata)

In [ ]:
# STEP 2 - Split the knowledge base into documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", ",", " "]
)
split_docs = splitter.split_documents(pages)

print(f"Total chunks: ", {len(split_docs)})

In [ ]:
# STEP 3 - Create embeddings and vector store using FAISS

from langchain_community.embeddings import BedrockEmbeddings
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv, find_dotenv
import boto3
from botocore.config import Config

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')

bedrock_client = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-east-1',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

embedding_model = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1", 
              client=bedrock_client)

vectors = FAISS.from_documents(split_docs, embedding=embedding_model)
print(vectors)

In [22]:
# STEP 4 - Integrate vector database with LLM using Retrieval chain mechanism

from langchain_core.prompts import PromptTemplate
from langchain_aws.chat_models import ChatBedrock
from langchain_classic.chains import RetrievalQA

custom_prompt_template = """
Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
--------------------------
{context}
Question: {question}
"""
CUSTOM_PROMPT = PromptTemplate(
    template=custom_prompt_template,
    input_variables=["context", "question"]
)
llm = ChatBedrock(model_id="amazon.nova-micro-v1:0", client=bedrock_client)
qa_chain = RetrievalQA.from_chain_type(llm,
    chain_type="stuff",
    retriever=vectors.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": CUSTOM_PROMPT}
)

In [ ]:
# STEP 5 - Build CLI based interface

# Loop to ask multiple questions
while True:
    question = input("\nEnter the question (or type 'exit' to quit): ")

    if question.lower() in ['exit', 'quit']:
        print("Exiting... Have a great day!")
        break

    response = qa_chain({"query": question})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Show the answer
    print("\nAnswer:", answer)
    print(f"Source documents: {source_documents}")

In [ ]:
# STEP 6 - User interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Gen-AI and Langchain based on the provided text."
)

# Launch the interface
iface.launch(share=True)